In [ ]:
import chromadb.config

class VectorStore:
    def __init__(self,path = ".\vectorstore",collection_name ="dsa_docs"):
        self.client = chromadb.Client(chromadb.config.Settings(persist_directory=path,anonymized_telemetry=False))
        self.collection = self.client.get_or_create_collection(name=collection_name)
    
    def add_chunks(self,chunks,embed_fn):
        existing = self.collection.count()
        if existing>0:
            print("Loading_existing")
        existing_ids = set(self.collection.get()["ids"])
        new_chunks,new_embeddings,new_ids = [],[],[]

        print("Building new vector store")
        for idx,chunk in enumerate(chunks):
            id_ = str(idx)
            if id_ not in existing_ids:
                new_chunks.append(chunk)
                new_embeddings.append(embed_fn(chunk))
                new_ids.append(id_)
        
        if new_chunks:
            self.collection.add(documents=new_chunks,embeddings=new_embeddings,ids=new_ids)
            print(f"Added{len(new_chunks)} new chunks")
        
    def search(self,query_vector,top_k=3):
        result = self.collection.query(query_embeddings=[query_vector],n_results=top_k)
        return result["documents"][0] if result["documents"] else []


In [22]:
from nltk.corpus import stopwords
from math import sqrt
#nltk.download("stopwords")

class DocumentLoader():
    def __init__(self,vector_store):
        self.chunks = []
        self.embedded_chunks = []
        self.text = ""
        self.vector_store = vector_store
        
    def load_file(self,path):
        with open(path) as f:
            self.text = f.read()
        return self.text
    
    def chunks_creation(self, size=200, overlap=50):
        self.chunks = []
        start = 0
        while start < len(self.text):
            self.chunks.append(self.text[start:start+size])
            start += size - overlap
        return self.chunks
    
    def embed_all_chunks(self,embed_fn):
        self.vector_store.add_chunks(self.chunks,embed_fn)
    
    def top_3_chunks(self,qurey_vector):
        return self.vector_store.search(qurey_vector,top_k=3)

In [ ]:
import ollama
from collections import OrderedDict,deque
import re

class ChatMemory:
    # -----------------------------------------------------------------------------------------------------------------------------------
    # 1.Initiation
    # -----------------------------------------------------------------------------------------------------------------------------------
    def __init__(self,model = "qwen2.5",system_prompt = "be a tutor",max_tokens=3000,cache_size=10,path=None):
        self.model = model
        self.max_tokens = max_tokens
        self.cache_size = cache_size
        self.chat_history = deque([{"role":"system","content":system_prompt}])
        self.cache = OrderedDict()
        self.ai_counter = 0
        self.cache_counter = 0
        self.stack = []
        
        self.vector_store = VectorStore(
            path="./vectorstore",
            collection_name="dsa_docs"
        )

        self.loader = DocumentLoader(self.vector_store)
        self.path = path
        self.doc_text = ''
    # -----------------------------------------------------------------------------------------------------------------------------------
    # 2.DOCUMENT PREPARATION
    # -----------------------------------------------------------------------------------------------------------------------------------
    # prepraing the documents by opening and converting them into chuncks    
    def prepare_document(self,path,size=200,overlap=50):
        self.doc_text = self.loader.load_file(path)
        self.loader.chunks_creation(size,overlap)
        self.loader.embed_all_chunks(self.embed)
        
    # -----------------------------------------------------------------------------------------------------------------------------------
    # 3.TEXT NORMALIZATION & KEYWORDS
    # -----------------------------------------------------------------------------------------------------------------------------------   
    # user input preprocessing and toaken calculation    
    def normalize(self,text):
        return re.sub(r'[^\w\s]','', text).strip().lower()
    
    # -----------------------------------------------------------------------------------------------------------------------------------
    # 4. EMBEDDING
    # -----------------------------------------------------------------------------------------------------------------------------------      
    # embedding the user input
    def embed(self,text):
        response = ollama.embeddings(model=self.model,prompt=text)
        return response["embedding"]
    
    # -----------------------------------------------------------------------------------------------------------------------------------
    # 5. CONTEXT RETRIVAL
    # -----------------------------------------------------------------------------------------------------------------------------------      
    #retriving the context where it is availabe in the doc or not
    def retrive_context(self,user_text):
        #keyword = self.extract_keywords(user_text)
        embedded_user = self.embed(user_text)
        top_chunks = self.loader.top_3_chunks(embedded_user)
        if not top_chunks:
            return "No Context Found"
        #return "\n\n".join(f"[score={c['score']}]\n{c['text']}"for c in top_chunks)
        combined = " ".join (top_chunks)
        if len(combined.split())<80:
            return "No Context Found"
        return "\n\n".join(top_chunks)
    
    # -----------------------------------------------------------------------------------------------------------------------------------
    # 6. TOKEN MANAGEMENT
    # -----------------------------------------------------------------------------------------------------------------------------------          
    def count_tokens(self,text):
        return len(text)//4
    
    def total_tokens(self):
        content = " ".join(x["content"] for x in self.chat_history)
        total_token_used = self.count_tokens(content)
        return total_token_used
    
    def trim_to_budget(self):
        while self.total_tokens()>self.max_tokens:
            del self.chat_history[1]
            del self.chat_history[1]
            
    # -----------------------------------------------------------------------------------------------------------------------------------
    # 7. CAHT HISTORY MANAGEMENT
    # -----------------------------------------------------------------------------------------------------------------------------------      
    #adding user input & reply to the history 
    def add_user(self, text):
        text = self.normalize(text)
        self.chat_history.append({"role":"user","content":text})
        self.trim_to_budget()
        return text
    
    def add_assistant(self,text):
        self.chat_history.append({"role":"assistant","content":text})
        
    def get_history(self):
        return list(self.chat_history)
    
    # -----------------------------------------------------------------------------------------------------------------------------------
    # 8.CACHE MANAGEMENT
    # -----------------------------------------------------------------------------------------------------------------------------------          
    def check_cache(self,text):
        if text in self.cache:
            self.cache_counter+=1
            return self.cache[text]
        else: return None
    
    def store_cache(self,text,ai_reply):
        if len(self.cache)>=self.cache_size:
            self.cache.popitem(last=False)
        self.cache[text] = ai_reply
        self.ai_counter+=1
        
    def get_stats(self):
        total = self.cache_counter + self.ai_counter
        if total == 0:
            return "hit rate: 0%"
        return f"hit rate: {(self.cache_counter / total) * 100:.2f}%"
    
    # -----------------------------------------------------------------------------------------------------------------------------------
    # 9. LLM CHAIN
    # -----------------------------------------------------------------------------------------------------------------------------------   
    # prompt chaining    
    def chain(self, user_, route_tool):
        # print(f"DEBUG chain received: {repr(user_)}")
        tool_name, tool_output = route_tool(user_, self)
        # print(f"DEBUG tool_name: {tool_name}")
        # print(f"DEBUG tool_output: {tool_output}")
        if tool_name in ["define", "calculate", "history"]:
            return tool_output
           
        context = self.retrive_context(user_)
        if context == "No Context Found":
            return "I can only answer the question about the document loaded. That topic is not covered"
        promt = f"""
        You are a tutor.
        IMPORTANT RULES:
        - Answer ONLY using the information present in the DOCUMENT CONTEXT.
        - Do NOT add new knowledge.
        - Do NOT infer beyond the document.
        - If the answer is not explicitly present, reply exactly:
        "Context not available in the provided document."
        DOCUMENT CONTEXT:
        {context}
        USER QUESTION:
        {user_}
        """
        r1 = ollama.chat(model=self.model,messages=[{"role":"user","content":promt}])
        base = r1["message"]["content"]
        self.stack.append(base)
        r2 = ollama.chat(model=self.model,messages=[{"role":"user","content":f"refine this :\n {base}"}])
        refined_ = r2["message"]["content"]
        self.stack.append(refined_)
        if "error" in refined_.lower() or len(refined_.strip())<5:
            self.stack.pop()
            r2 = ollama.chat(model=self.model,messages=[{"role":"user","content":f"refine this :\n {base}"}])
            refined_ = r2["message"]["content"]
        return refined_
    
    # -----------------------------------------------------------------------------------------------------------------------------------
    # 10. MAIN CHAT ENTRY POINT
    # -----------------------------------------------------------------------------------------------------------------------------------   
    # main function to combine all
    def chat(self,user_,route_tool):
        normalized = self.normalize(user_)
        self.chat_history.append({"role":"user","content":user_})
        self.trim_to_budget()
        cached = self.check_cache(normalized)
        if cached:
            reply = cached
        else:
            reply = self.chain(user_,route_tool)
            self.store_cache(normalized,reply)
            
        self.add_assistant(reply)
        return reply 

## agent function

In [ ]:
import math
#--------------------------------------------------------------------------------------------------------------------------------------------
#TOOL 1: search document
#--------------------------------------------------------------------------------------------------------------------------------------------
def search_document(query,memory):
    return memory.retrive_context(query)

#--------------------------------------------------------------------------------------------------------------------------------------------
#TOOL 2: calculate
#--------------------------------------------------------------------------------------------------------------------------------------------
def calculate(expression):
   try:
       allowed = "0123456789+-*/(). eE"
       cleaned = expression.replace(" ", "")
    #    print(f"DEBUG expression: {repr(cleaned)}")  # add this
       if any(ch not in allowed for ch in cleaned):
           bad = [ch for ch in cleaned if ch not in allowed]
        #    print(f"DEBUG blocked chars: {bad}")  # add this
           return "unsafe expression"
       result = eval(expression, {"__builtins__": None}, vars(math))
    #    print(f"DEBUG result: {result}")  # add this
       return str(result)
   except Exception as e:
       return f"calculation error: {str(e)}"
    
# -------------------------------
# TOOL 3: define_term
# -------------------------------
def define_term(term):
    definitions = {
        "array": "A linear data structure storing elements in contiguous memory.",
        "graph": "A set of nodes connected by edges; can be directed or undirected.",
        "queue": "A FIFO (first-in-first-out) data structure.",
        "stack": "A LIFO (last-in-first-out) data structure.",
        "tree": "A hierarchical data structure with a root and child nodes.",
    }
    term = term.lower().strip()
    return definitions.get(term, "Definition not found.")

# -------------------------------
# TOOL 4: get_history
# -------------------------------
def get_history(memory):
    history = list(memory.chat_history)[1:]
    return "\n".join(f"{m['role']}: {m['content']}" for m in history)

TOOLS = {
    "search": search_document,
    "calculate": calculate,
    "define": define_term,
    "history": get_history
}

In [25]:
def route_tool(user_input, memory):
   #print(f"DEBUG original: {repr(user_input)}")
   # --- Math check FIRST before any cleaning
   if "**" in user_input:
    #    print("DEBUG: going to calculate **")
       return "calculate", TOOLS["calculate"](user_input)
   math_symbols = ["+", "-", "*", "/", "^", "(", ")"]
   if any(sym in user_input for sym in math_symbols):
    #    print("DEBUG: going to calculate symbols")
       return "calculate", TOOLS["calculate"](user_input)
   # --- Now clean for other tools
   u = user_input.lower().strip()
   cleaned = re.sub(r'[^\w\s]', '', u)
   stopwords = {"a", "an", "the"}
   words = [w for w in cleaned.split() if w not in stopwords]
   cleaned = " ".join(words)
#    print(f"DEBUG cleaned: {repr(cleaned)}")
#    print(f"DEBUG u: {repr(u)}")
   # --- Definition tool
   definition_keywords = ["what is", "define"]
   if any(k in u for k in definition_keywords):
    #    print("DEBUG define check: True")
       # extract from u directly
       term = u
       for k in definition_keywords:
           term = term.replace(k, "")
       # clean punctuation and stopwords after extraction
       term = re.sub(r'[^\w\s]', '', term)
       term = " ".join(w for w in term.split()
                      if w not in {"a", "an", "the"})
       term = term.strip()
    #    print(f"DEBUG term extracted: {repr(term)}")
       return "define", TOOLS["define"](term)
   # --- History tool
   history_keywords = ["history", "previous", "discuss", "talk", "recap"]
   if any(h in u for h in history_keywords):
       return "history", TOOLS["history"](memory)
   # --- Default: Document search
   return "search", TOOLS["search"](user_input, memory)

In [27]:
memory = ChatMemory()
memory.prepare_document(
    path=r"E:\training\logger\text.txt",
    size=200,
    overlap=50)
# memory.loader.embed_all_chunks(model='qwen2.5')
#print(memory.retrive_context("binary search"))
while True:
    user = input("chat: ")
    print("user :",user)
    if user.lower() == "quit":
        print(memory.get_stats())
        break
    reply = memory.chat(user,route_tool)
    print("qwen2.5:", reply)

Loading_existing
Building new vector store
user : what is datastructure and algorithm
qwen2.5: Definition not found.
user : explain data structure and algorithm
qwen2.5: A data structure is a method for organizing and storing data to facilitate efficient management and retrieval of information. An algorithm, on the other hand, is a well-defined sequence of steps designed to solve specific problems using these data structures.

Common data structures such as stacks, queues, hash tables, trees, heaps, and graphs each serve unique purposes and are selected based on the nature of the problem at hand:
- Stacks and queues manage hierarchical relationships or follow First-In-First-Out (FIFO) principles.
- Hash tables enable fast lookups by allowing direct access to elements using a key.
- Trees and heaps are used for managing hierarchical data, with trees often representing nested structures, and heaps providing an efficient way to maintain order without full sorting.
- Graphs represent netwo